# Yara new indexation — KMeans monthly clustering

This notebook reuses the existing preprocessing pipeline:
- `compute_bands_all()` for loading bands + indices + Lyzenga correction
- `prepare_data()` for resampling, valid-pixel masking, percentile clipping, and scaling
- monthly RGB creation and side-by-side monthly PNG export

It trains **KMeans with 10 clusters** on the baseline period, then predicts monthly labels for all months using the same scaler.

In [9]:
import os

print(os.listdir(r"C:\Users\yaroc\OneDrive\Документы\GitHub\RemoteSensing_CoralReefs\scripts"))

['cache.py', 'config.py', 'data_loading.py', 'extra', 'features.py', 'lyzenga.py', 'model.py', 'validation.py', 'visualization.py', '__init__.py', '__pycache__']


In [11]:
import os
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from os.path import join as pjoin
from sklearn.cluster import KMeans
import sys

PROJECT_ROOT = r"C:/Users/yaroc/OneDrive/Документы/GitHub/RemoteSensing_CoralReefs"
repo_dir = r"C:/Users/yaroc/OneDrive/Документы/GitHub/RemoteSensing_CoralReefs"
sys.path.append(PROJECT_ROOT)

import scripts.config as config
from scripts.data_loading import load_stack, compute_bands_all
from scripts.features import prepare_data
from scripts.model import save_labels_timeseries, print_cluster_distribution
from scripts.visualization import load_monthly_rgb, get_all_monthly_timestamps, percentile_stretch


ModuleNotFoundError: No module named 'scripts.config'

## 1. Settings

In [ ]:
# KMeans-specific settings
KMEANS_K = 10
KMEANS_RANDOM_STATE = 42
KMEANS_N_INIT = 20

# Reuse same baseline / resampling logic as the GMM pipeline
BASELINE_YEAR = config.BASELINE_YEAR
RESAMPLE_FREQ = config.RESAMPLE_FREQ
RESAMPLE_AGG = config.RESAMPLE_AGG

# Names for saved outputs
YEAR_STRING = ','.join(str(y) for y in BASELINE_YEAR)
RUN_NAME_KMEANS = f"B({YEAR_STRING})_kmeans{KMEANS_K}_{RESAMPLE_FREQ}_{RESAMPLE_AGG}{'_bathy' if config.LYZENGA_ALG else ''}"

MODEL_NAME = f"kmeans_{RUN_NAME_KMEANS}"
LABELS_DIR_NAME = f"labels_monthly_{RUN_NAME_KMEANS}"
PNG_DIR = pjoin(config.FIGURES_DIR, f"monthly_kmeans_pngs_{RUN_NAME_KMEANS}")
os.makedirs(PNG_DIR, exist_ok=True)

print('RUN_NAME_KMEANS:', RUN_NAME_KMEANS)
print('PNG output folder:', PNG_DIR)


RUN_NAME_KMEANS: B(2022,2023)_kmeans10_MS_median_bathy
PNG output folder: C:\Users\yaroc\OneDrive\Документы\GitHub\RemoteSensing_CoralReefs\reports\figures\monthly_kmeans_pngs_B(2022,2023)_kmeans10_MS_median_bathy


## 2. Helper function for side-by-side monthly PNG export

In [ ]:
def save_monthly_comparisons_kmeans(monthly_rgb, spatial_labels_monthly, timestamps, all_timestamps, output_dir, from_year=None, n_clusters=10):
    """Save one PNG per month: original RGB on the left, KMeans clusters on the right."""
    os.makedirs(output_dir, exist_ok=True)
    cmap = plt.cm.get_cmap('tab10', n_clusters)
    cmap.set_bad(color='black')

    if from_year is not None:
        timestamps_to_plot = [ts for ts in timestamps if pd.Timestamp(ts).year >= from_year]
    else:
        timestamps_to_plot = list(timestamps)

    for ts in tqdm(timestamps_to_plot, desc='Saving monthly KMeans comparisons'):
        spatial_idx = all_timestamps.index(ts)
        year = pd.Timestamp(ts).year
        month = pd.Timestamp(ts).month
        ts_str = pd.Timestamp(ts).strftime('%Y_%m')

        rgb = monthly_rgb.sel(
            time=(monthly_rgb.time.dt.year == year) & (monthly_rgb.time.dt.month == month)
        ).squeeze().values.transpose(1, 2, 0)

        fig, axes = plt.subplots(1, 2, figsize=(24, 10))

        axes[0].imshow(np.flipud(percentile_stretch(rgb)), origin='upper')
        axes[0].set_title(f'{ts_str} - Original RGB', fontsize=14)
        axes[0].axis('off')

        im = axes[1].imshow(
            np.flipud(spatial_labels_monthly[spatial_idx]),
            cmap=cmap,
            vmin=-0.5,
            vmax=n_clusters - 0.5,
            interpolation='nearest',
            origin='upper'
        )
        axes[1].set_title(f'{ts_str} - KMeans ({n_clusters} clusters)', fontsize=14)
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1], label='Cluster ID', ticks=range(n_clusters))

        plt.tight_layout()
        plt.savefig(pjoin(output_dir, f'kmeans_compare_{ts_str}.png'), dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f'All monthly PNGs saved to: {output_dir}')


## 3. Load full stack and compute all features

This reuses the original preprocessing logic, including:
- raw band loading
- Lyzenga correction when enabled in `config.py`
- spectral indices


In [ ]:
import glob 
import rioxarray
import xarray as xr
def load_stack_chunked(repo_dir, x_chunk=512, y_chunk=512):
    pattern1 = os.path.join(repo_dir, "data", "raw", "PSScene", "*_3B_AnalyticMS_SR_8b_clip_philip.tif")
    pattern2 = os.path.join(repo_dir, "data", "raw", "PSScene", "*_3B_AnalyticMS_SR_8b_clip_tricia.tif")
    pattern3 = os.path.join(repo_dir, "data", "raw", "PSScene", "*_3B_AnalyticMS_SR_8b_clip_dennis.tif")

    files = sorted(
        glob.glob(pattern1) +
        glob.glob(pattern2) +
        glob.glob(pattern3)
    )

    dates = [pd.Timestamp(os.path.basename(f)[:8]) for f in files]

    arrays = []
    for f, date in zip(files, dates):
        da = rioxarray.open_rasterio(
            f,
            chunks={"band": 1, "x": x_chunk, "y": y_chunk}
        )
        da = da.assign_coords(time=date)
        arrays.append(da)

    stack = xr.concat(arrays, dim="time").sortby("time")
    return stack

Storing band values

In [ ]:
def extract_bands_from_stack(stack):
    band_map = {
        "cb": 1,
        "blue": 2,
        "green": 4,
        "yellow": 5,
        "red": 6,
        "nir": 8
    }

    bands = {}
    for name, band_idx in band_map.items():
        bands[name] = stack.sel(band=band_idx)

    return bands

Indices calculation

In [ ]:
def safe_divide(a, b):
    return xr.where(b != 0, a / b, np.nan)

def calculate_indices_from_bands(bands):
    indices = {}

    indices["blue_green"] = safe_divide(bands["coastal_blue"], bands["green"])
    indices["brightness"] = (bands["red"] + bands["green"] + bands["blue"] + bands["yellow"] + bands["cb"]) / 5
    indices["ndavi_reversed"] = safe_divide(
        bands["cb"] - bands["nir"],
        bands["nir"] + bands["cb"]
    )

    return indices

full feature assembly while keeping raw bands + indices + Lyzenga

In [ ]:
from scripts.lyzenga import (
    apply_lyzenga,
    dem_resampling,
    depth_regression_data,
)
import scripts.config as config
import rioxarray

def compute_bands_all_from_stack(stack):
    bands = extract_bands_from_stack(stack)
    bands_all = dict(bands)

    if config.LYZENGA_ALG:
        # load DEM
        dem = rioxarray.open_rasterio(config.DEM_FILE, chunks={"x": 512, "y": 512})

        # resample DEM to imagery
        dem_resampled = dem_resampling(bands, dem)

        # prepare regression pixels + bathy
        regression_pixels, bathy = depth_regression_data(bands, dem_resampled)

        # you need deep water values for Lyzenga
        dw_values = config.LYZENGA_DEEP_WATER

        # apply Lyzenga correctly
        lyzenga_bands = apply_lyzenga(
            raw_bands=bands,
            dw_values=dw_values,
            regression_pixels=regression_pixels,
            use_bathy=True,
            bathy=bathy
        )
        bands_all.update(lyzenga_bands)

    spectral_indices = calculate_indices_from_bands(bands)
    bands_all.update(spectral_indices)

    return bands_all

ModuleNotFoundError: No module named 'scripts.lyzenga'

In [ ]:
bands_all = compute_bands_all_from_stack(stack)

feature_names = list(bands_all.keys())
print("Feature names used for clustering:")
print(feature_names)
print(f"Total features: {len(feature_names)}")

RasterioIOError: OTI_25cm_Q820_all_bathy_topo_DTM_refract_-1m_GM(bin,min,0.25,1).tif: No such file or directory

## 4. Prepare baseline data for training

This step reuses `prepare_data()`, so it keeps the same:
- monthly resampling
- valid pixel mask
- percentile clipping
- standardization


In [ ]:
baseline_data, baseline_mask, scaler = prepare_data(
    bands_all=bands_all,
    year=BASELINE_YEAR,
    resample_freq=RESAMPLE_FREQ,
    resample_agg=RESAMPLE_AGG,
    scaler=None,
    lower=config.PERCENTILE_LOWER,
    upper=config.PERCENTILE_UPPER,
)

print('Baseline normalized shape:', baseline_data.shape)
print('Valid baseline pixels:', baseline_mask.sum())


NameError: name 'bands_all' is not defined

## 5. Train KMeans (10 clusters)

In [ ]:
kmeans = KMeans(
    n_clusters=KMEANS_K,
    random_state=KMEANS_RANDOM_STATE,
    n_init=KMEANS_N_INIT
)

print('Training KMeans...')
kmeans.fit(baseline_data)
print('KMeans training complete')

joblib.dump(kmeans, pjoin(config.MODELS_DIR, f'{MODEL_NAME}.pkl'))
joblib.dump(scaler, pjoin(config.MODELS_DIR, f'{MODEL_NAME}_scaler.pkl'))
print('Saved model and scaler')


## 6. Prepare all monthly data using the same scaler

In [ ]:
monthly_data, monthly_mask, _ = prepare_data(
    bands_all=bands_all,
    year=None,
    resample_freq=RESAMPLE_FREQ,
    resample_agg=RESAMPLE_AGG,
    scaler=scaler,
    lower=config.PERCENTILE_LOWER,
    upper=config.PERCENTILE_UPPER,
)

print('Monthly normalized shape:', monthly_data.shape)
print('Valid monthly pixels:', monthly_mask.sum())


## 7. Predict monthly KMeans labels

In [ ]:
labels_monthly = kmeans.predict(monthly_data)
print_cluster_distribution(labels_monthly)

np.save(pjoin(config.INTERIM_DIR, f'{MODEL_NAME}_labels.npy'), labels_monthly)
np.save(pjoin(config.INTERIM_DIR, f'{MODEL_NAME}_mask.npy'), monthly_mask)
np.save(pjoin(config.INTERIM_DIR, f'{MODEL_NAME}_data.npy'), monthly_data)
print('Saved monthly labels, mask, and monthly data arrays')


## 8. Convert labels back into monthly spatial maps

In [ ]:
all_timestamps = get_all_monthly_timestamps(bands_all)
num_timesteps = len(all_timestamps)
reference_band = bands_all[list(bands_all.keys())[0]]

spatial_labels_monthly = save_labels_timeseries(
    labels=labels_monthly,
    mask=monthly_mask,
    reference_band=reference_band,
    name=LABELS_DIR_NAME,
    num_timesteps=num_timesteps,
    timestamps=all_timestamps,
)

np.save(pjoin(config.INTERIM_DIR, f'{LABELS_DIR_NAME}.npy'), spatial_labels_monthly)
print('Saved spatial monthly labels array')
print('Spatial label array shape:', spatial_labels_monthly.shape)


## 9. Save one PNG per month: original RGB vs KMeans result

In [ ]:
save_monthly_comparisons_kmeans(
    monthly_rgb=monthly_rgb,
    spatial_labels_monthly=spatial_labels_monthly,
    timestamps=all_timestamps,
    all_timestamps=all_timestamps,
    output_dir=PNG_DIR,
    from_year=None,
    n_clusters=KMEANS_K,
)


## 10. Quick check for one month

In [ ]:
example_idx = 0
example_ts = pd.Timestamp(all_timestamps[example_idx]).strftime('%Y-%m')

rgb = monthly_rgb.sel(time=all_timestamps[example_idx], method='nearest').squeeze().values.transpose(1, 2, 0)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(np.flipud(percentile_stretch(rgb)), origin='upper')
axes[0].set_title(f'{example_ts} - Original RGB')
axes[0].axis('off')

im = axes[1].imshow(np.flipud(spatial_labels_monthly[example_idx]), cmap=plt.cm.get_cmap('tab10', KMEANS_K), vmin=-0.5, vmax=KMEANS_K-0.5)
axes[1].set_title(f'{example_ts} - KMeans ({KMEANS_K} clusters)')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], ticks=range(KMEANS_K), label='Cluster ID')
plt.tight_layout()
plt.show()
